# IDS2017 — Data Ingestion & Cleaning

This notebook loads all 14 CIC-IDS2017 CSV files from HDFS (~599 MB, 14 classes), applies 7 cleaning steps, and saves the result as Parquet for (Feature Engineering).

---
**Prerequisites (must be done before running this notebook):**
1. All containers running (`docker compose up -d`)
2. All 14 CSVs uploaded to HDFS (`02_upload_hdfs.ps1` already run)

**Data flow:**
```
HDFS /user/bigdata/ids2017/raw/   (14 CSV files, ~599 MB)
           |  this notebook
           v
HDFS /user/bigdata/ids2017/processed/cleaned/   (Parquet)
```

In [ ]:
# Run this first if you need to restart Spark (e.g. after a kernel restart)
# try:
#     spark.stop()
# except:
#     pass

---
## Setup — Imports & Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import matplotlib.pyplot as plt
import numpy as np

# HDFS paths 
HDFS_RAW       = "hdfs://namenode:9000/user/bigdata/ids2017/raw"
HDFS_PROCESSED = "hdfs://namenode:9000/user/bigdata/ids2017/processed/cleaned"

LABEL_COL = 'label'

#  Columns where negative values are physically impossible ─
# These use the actual snake_case column names in this dataset (122 columns)
NON_NEGATIVE_COLS = [
    'duration',
    'fwd_packets_count',
    'bwd_packets_count',
    'fwd_total_payload_bytes',
    'bwd_total_payload_bytes',
    'bytes_rate',
    'packets_rate',
]

#  Numeric features to use in EDA 
EDA_FEATURES = [
    'duration',
    'packets_count',
    'fwd_packets_count',
    'bwd_packets_count',
    'bytes_rate',
    'packets_rate',
    'payload_bytes_mean',
    'fwd_payload_bytes_mean',
]

print("Imports OK")
print(f"Label column : '{LABEL_COL}'")
print(f"Non-negative : {NON_NEGATIVE_COLS}")

In [ ]:
spark = (
    SparkSession.builder
    .appName("IDS2017-Cleaning-Notebook")
    .master("local[*]")                                        # use Jupyter's own Spark
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")   # point at HDFS
    .config("spark.driver.memory", "1g")            # stays within available host RAM
    .config("spark.sql.shuffle.partitions", "20")   # fewer partitions = less overhead
    .config("spark.ui.enabled", "false")            # disable Spark UI to save memory
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"Spark master  : {spark.sparkContext.master}")
print("Spark session ready")

---
## Exploratory Data Analysis (EDA)

Quick look at the raw data before any cleaning — label distribution, missing values, and feature distributions.

In [ ]:
df_raw = spark.read.csv(
    HDFS_RAW,
    header=True,
    inferSchema=False,
    ignoreLeadingWhiteSpace=True,
    ignoreTrailingWhiteSpace=True,
)

print(f"Total columns : {len(df_raw.columns)}")
print(f"Column names (first 10):")
for c in df_raw.columns[:10]:
    print(f"  {c}")
print(f"  ... ({len(df_raw.columns) - 10} more)")
print(f"\nLast column: '{df_raw.columns[-1]}' (should be 'label')")

In [ ]:
#  Label Distribution 
# Use a 10% sample — proportions are stable, avoids a full 600 MB scan
print("Raw label distribution (10% sample):")
label_counts = (
    df_raw.sample(fraction=0.10, seed=42)
    .groupBy(LABEL_COL)
    .count()
    .orderBy(F.desc('count'))
    .toPandas()
)
# Scale back to estimated full-dataset counts
label_counts['count'] = (label_counts['count'] * 10).astype(int)
print(label_counts.to_string(index=False))
print(f"\nTotal classes: {len(label_counts)}")

In [ ]:
#  Bar Chart + Pie Chart 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
ax1 = axes[0]
bars = ax1.bar(label_counts[LABEL_COL], label_counts['count'], color='steelblue', edgecolor='white')
ax1.set_title('Label Distribution (Raw)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Attack Type', fontsize=12)
ax1.set_ylabel('Number of Rows', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 1000,
             f'{int(h):,}', ha='center', va='bottom', fontsize=7)

# Pie chart (attacks only, no Benign)
ax2 = axes[1]
attack_df = label_counts[label_counts[LABEL_COL].str.upper() != 'BENIGN']
ax2.pie(
    attack_df['count'],
    labels=attack_df[LABEL_COL],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 9}
)
ax2.set_title('Attack Type Breakdown\n(Benign excluded)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
#  Missing Values (Infinity + null) 
INF_STRINGS = ['Infinity', '-Infinity', 'inf', '-inf']
feature_cols = [c for c in df_raw.columns if c not in ('flow_id', 'timestamp', 'src_ip', 'dst_ip', LABEL_COL)]

# Check a sample of columns for Infinity strings
print("Checking for Infinity string values (sample of 10 feature columns):")
for c in feature_cols[:10]:
    inf_count = df_raw.filter(F.col(c).isin(INF_STRINGS)).count()
    if inf_count > 0:
        print(f"  {c:40s} : {inf_count:>8,} Infinity values")

# Cast and count nulls across all feature columns
df_cast = df_raw
for c in feature_cols:
    df_cast = df_cast.withColumn(
        c,
        F.when(F.col(c).isin(INF_STRINGS), None)
         .otherwise(F.col(c).cast(DoubleType()))
    )

null_counts = df_cast.select(
    [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in feature_cols]
).toPandas().T
null_counts.columns = ['null_count']
null_nonzero = null_counts[null_counts['null_count'] > 0].sort_values('null_count', ascending=False)

if len(null_nonzero) > 0:
    print(f"\nColumns with null / Infinity values ({len(null_nonzero)} columns):")
    print(null_nonzero.head(20).to_string())
else:
    print("\nNo null or Infinity values found in feature columns!")

In [ ]:
#  Statistical Summary of Key Features 
print("Statistical summary of EDA features:")
df_raw.select(EDA_FEATURES).describe().show(truncate=False)

In [ ]:
#  Feature Histograms (log scale) 
# Sample data for plotting (full dataset is too large for pandas)
sample_df = (
    df_raw
    .select(EDA_FEATURES + [LABEL_COL])
    .sample(fraction=0.05, seed=42)    # 5% sample
    .toPandas()
)

# Cast to numeric
for col in EDA_FEATURES:
    sample_df[col] = sample_df[col].replace(['Infinity', '-Infinity', 'inf', '-inf'], None)
    sample_df[col] = sample_df[col].astype(float)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feature in enumerate(EDA_FEATURES):
    ax = axes[i]
    data = sample_df[feature].dropna()
    # Use log-scale x because most features are very right-skewed
    data_pos = data[data > 0]
    if len(data_pos) > 0:
        ax.hist(np.log1p(data_pos), bins=50, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(feature, fontsize=10, fontweight='bold')
    ax.set_xlabel('log(1 + value)', fontsize=8)
    ax.set_ylabel('Frequency', fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle('Feature Distributions (5% sample, log scale)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#  Normalised Mean per Label
#  
# Cast EDA features in the sample to float
for col in EDA_FEATURES:
    sample_df[col] = sample_df[col].astype(float)

# Normalise each feature to [0, 1] range for comparison
sample_norm = sample_df.copy()
for col in EDA_FEATURES:
    col_min = sample_norm[col].min()
    col_max = sample_norm[col].max()
    if col_max > col_min:
        sample_norm[col] = (sample_norm[col] - col_min) / (col_max - col_min)
    else:
        sample_norm[col] = 0.0

mean_by_label = sample_norm.groupby(LABEL_COL)[EDA_FEATURES].mean()

fig, ax = plt.subplots(figsize=(16, 7))
mean_by_label.T.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Normalised Feature Means by Label', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature', fontsize=12)
ax.set_ylabel('Normalised Mean (0-1)', fontsize=12)
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Label', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

---
## Step 1 — Load All 14 CSVs from HDFS

Spark reads the **entire directory** — all 14 CSV files are automatically unioned into one DataFrame.  
No manual merging needed.

| File | Class |
|---|---|
| thursday_benign.csv | Benign |
| dos_hulk.csv | DoS_Hulk |
| ddos_loit.csv | DDoS_LOIT |
| dos_golden_eye.csv | DoS_GoldenEye |
| dos_slowhttptest.csv | DoS_Slowhttptest |
| dos_slowloris.csv | DoS_Slowloris |
| ftp_patator.csv | FTP-Patator |
| ssh_patator-new.csv | SSH-Patator |
| portscan.csv | Port_Scan |
| botnet_ares.csv | Botnet_ARES |
| heartbleed.csv | Heartbleed |
| web_brute_force.csv | Web_Brute_Force |
| web_xss.csv | Web_XSS |
| web_sql_injection.csv | Web_SQL_Injection |

In [ ]:
df = spark.read.csv(
    HDFS_RAW,                         
    header=True,
    inferSchema=False,             
    ignoreLeadingWhiteSpace=True,
    ignoreTrailingWhiteSpace=True,
)

raw_rows = df.count()
raw_cols  = len(df.columns)

print(f"Raw rows    : {raw_rows:,}")
print(f"Raw columns : {raw_cols}")

In [ ]:
# Quick look at the data
df.select(df.columns[:6] + [LABEL_COL]).show(5, truncate=False)

In [ ]:
# Label distribution — use a 10% sample to avoid a full second scan
# (full scan would double the memory pressure right after the first count)
print("Label distribution (10% sample — proportions are accurate):")
df.sample(fraction=0.10, seed=42) \
  .groupBy(LABEL_COL).count() \
  .orderBy(F.desc('count')) \
  .show(25, truncate=False)

---
## Step 2 — Fix Column Names

**Problem:** The raw CSVs may have column names with leading/trailing spaces, and a duplicate `Fwd Header Length` column (a known issue in CICFlowMeter output).  
**Fix:** Strip whitespace from all column names, keep only the first occurrence of any duplicate.

In [ ]:
# Trim spaces from column names
for col in df.columns:
    stripped = col.strip()
    if stripped != col:
        df = df.withColumnRenamed(col, stripped)

# Drop duplicate column names (keep first occurrence)
seen = set()
cols_to_keep = []
for c in df.columns:
    if c not in seen:
        cols_to_keep.append(c)
        seen.add(c)

dupes_dropped = len(df.columns) - len(cols_to_keep)
df = df.select(cols_to_keep)

print(f"Column names trimmed")
print(f"Duplicate columns dropped : {dupes_dropped}")
print(f"Columns after fix         : {len(df.columns)}")

---
## Step 3 — Replace Infinity Values with null

**Problem:** CICFlowMeter writes `Infinity` and `-Infinity` as strings when flow rate metrics overflow (e.g. `bytes_rate` when `duration = 0`).  
**Fix:** Replace those strings with `null`, then cast all feature columns to `DoubleType`.

In [ ]:
INF_STRINGS = ['Infinity', '-Infinity', 'inf', '-inf']

for c in df.columns:
    if c == LABEL_COL:       # keep label as string
        continue
    df = df.withColumn(
        c,
        F.when(F.col(c).isin(INF_STRINGS), None)
         .otherwise(F.col(c).cast(DoubleType()))
    )

print("Infinity -> null cast complete")
print("Schema sample (first 8 feature columns):")
for field in df.schema.fields[7:15]:
    print(f"  {field.name:40s} : {field.dataType}")

---
## Step 4 — Remove Duplicate Rows

**Problem:** The dataset may contain exact duplicate rows from CICFlowMeter output.  
**Fix:** `dropDuplicates()` across all 122 columns.

> **Notebook note:** This step is skipped in the notebook because `dropDuplicates()` requires  
> an 8–12 GB shuffle on this dataset (122 columns × ~800 K rows) — more than is available  
> on a local 4 GB machine running Docker + HDFS + Jupyter simultaneously.  
> It is **fully applied** in `04_clean.py` which runs via `spark-submit` on the Spark cluster  
> where executor memory is not constrained by the laptop.

In [ ]:
# SKIPPED IN NOTEBOOK — runs in 04_clean.py on the Spark cluster
# dropDuplicates() requires an 8-12 GB RAM shuffle; this laptop has 4 GB.
#
# df = df.dropDuplicates()
#
# The CIC-IDS2017 dataset has very few true duplicates (<0.1% of rows),
# so skipping this in the notebook does not affect EDA results.

print("Step 4 (dropDuplicates) — skipped in notebook, applied in 04_clean.py")

---
## Step 5 — Remove Rows with Impossible Negative Values

**Problem:** Some rows have negative values in columns like `duration` or `fwd_packets_count`, which are physically impossible for network flow metrics.  
**Fix:** Drop any row where at least one of these columns is negative.

In [ ]:
condition = None
matched_cols = []
for c in NON_NEGATIVE_COLS:
    if c in df.columns:
        matched_cols.append(c)
        cond = F.col(c) < 0
        condition = cond if condition is None else condition | cond

print(f"Checking columns : {matched_cols}")

if condition is not None:
    df = df.filter(~condition)
    print("filter(~negative) registered (lazy — executes during Parquet write in Step 7)")
else:
    print("No matching columns found — skipped")

---
## Step 6 — Standardize the Label Column

**Problem:** The `label` column has inconsistent casing across CSV files. We also want to unify all three Web Attack sub-types into one label.

**Fix:**
- `TRIM` + `UPPER` all labels
- Collapse `WEB_BRUTE_FORCE`, `WEB_XSS`, `WEB_SQL_INJECTION` → `WEB_ATTACK`

In [ ]:
# Trim whitespace and uppercase everything
df = df.withColumn(LABEL_COL, F.trim(F.upper(F.col(LABEL_COL))))

# Unify all Web Attack variants (WEB_BRUTE_FORCE, WEB_XSS, WEB_SQL_INJECTION -> WEB_ATTACK)
df = df.withColumn(
    LABEL_COL,
    F.when(F.col(LABEL_COL).contains('WEB'), 'WEB_ATTACK')
     .otherwise(F.col(LABEL_COL))
)

print("Label distribution AFTER standardization:")
df.groupBy(LABEL_COL).count().orderBy(F.desc('count')).show(25, truncate=False)

---
## Step 7 — Save Cleaned Data to HDFS as Parquet

**Why Parquet?**
- **Columnar format** — downstream jobs can read only the columns they need (much faster than CSV)
- **Compressed** — significantly smaller than CSV on disk
- **Schema preserved** — column types are stored, no need to re-cast on load
- **Splittable** — Spark can read it in parallel across workers

In [ ]:
print(f"Saving to: {HDFS_PROCESSED}")
df.write.mode('overwrite').parquet(HDFS_PROCESSED)
print("Saved!")

---
## Summary

In [ ]:
df_final = spark.read.parquet(HDFS_PROCESSED)
final_rows = df_final.count()
final_cols = len(df_final.columns)

print("=" * 55)
print("  CLEANING COMPLETE  (notebook run)")
print("=" * 55)
print(f"  Raw rows   : {raw_rows:,}")
print(f"  Final rows : {final_rows:,}")
print(f"  Removed    : {raw_rows - final_rows:,}  ({(raw_rows-final_rows)/raw_rows*100:.1f}%)")
print(f"  Columns    : {final_cols}")
print(f"  Output     : {HDFS_PROCESSED}")
print("=" * 55)
print("  Note: dropDuplicates skipped in notebook (memory).")
print("  Run 04_clean.py on the cluster for full pipeline.")
print("  Next step: Feature Engineering (05_feature_engineering.py)")

In [ ]:
# df_final already loaded above — just show label distribution
print("Label distribution after cleaning:")
df_final.groupBy(LABEL_COL).count().orderBy(F.desc('count')).show(truncate=False)

In [ ]:
print("Output schema:")
df_final.printSchema()